# Preprocesamiento de Datos

**Objetivo principal** : Preparar los datos de los subyacentes y las cadenas de opciones para poder calcular precios teóricos de opciones mediante Monte Carlo, Árbol Binomial y Black-Scholes. Esto incluye limpieza, filtrado, cálculo de parámetros históricos y selección de los contratos apropiados.

## Preparación del dataset de subyacentes

In [1]:
import pandas as pd
df_hist = pd.read_pickle("../data/historico_desde_2024_trabajo_MNFI.pkl")

In [2]:
df_hist.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 531 entries, 2024-01-02 to 2026-01-21
Columns: 162 entries, ('Adj Close', 'AAPL') to ('Volume', 'WBD')
dtypes: float64(162)
memory usage: 676.2 KB


In [3]:
df_hist.columns.levels[0]


Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

Imprimimos de antemano información general del dataset que nos han facilitado, observamos que tiene datos desde `2024-01-02` a `2026-01-21`.

Para cada activo se nos da información de `Adj Close` , `Close`,`High`,`Low`,`Open` y`Volume`:
- **Adjusted Close Price**: Es el precio de cierre de la acción, pero ajustado por dividendos, splits y otros eventos corporativos. Clave para estimar rentabilidades y volatilidad histórica.

- **Close**: Precio de cierre sin ajustes. Representa el último precio al que se negoció el activo durante la sesión. 

- **Open**: Precio al que el activo comenzó a cotizar al inicio de la sesión.  

- **High**: Precio máximo alcanzado durante la sesión.  
  Indica niveles de resistencia y la presión compradora durante el día.

- **Low**:Precio mínimo alcanzado durante la sesión.  

- **Volume** : Volumen de operaciones del día de dicho activo, con dicho dato podemos observar cambio de tendencia o liquidez




In [4]:

nan_totales = df_hist.isna().sum().sum()  
nan_por_col = df_hist.isna().sum()

nan_top10 = nan_por_col.sort_values(ascending=False).head(10)


duplicados = df_hist.duplicated().sum()
filas_duplicadas = df_hist[df_hist.duplicated(keep=False)] if duplicados > 0 else pd.DataFrame()


if isinstance(df_hist.index, pd.DatetimeIndex):
    fechas_esperadas = pd.date_range(start=df_hist.index.min(), end=df_hist.index.max(), freq='B')
    fechas_faltantes = fechas_esperadas.difference(df_hist.index)
else:
    fechas_faltantes = []

'''
print("="*50)
print("INTEGRIDAD DE DATOS - RESUMEN")
print("="*50)
print(f"Total de valores NaN en el DataFrame: {nan_totales}\n")

if not nan_top10.empty:
    print("Top 10 columnas con más valores NaN:")
    print(nan_top10)
else:
    print("No hay valores NaN en ninguna columna.\n")

print("-"*50)
print(f"Número de filas duplicadas: {duplicados}")
if duplicados > 0:
    print("Filas duplicadas:")
    print(filas_duplicadas)
print("-"*50)

if len(fechas_faltantes) > 0:
    print(f"Fechas faltantes ({len(fechas_faltantes)}):")
    print(fechas_faltantes)
else:
    print("No hay fechas faltantes.\n")
print("="*50)
'''


'\nprint("="*50)\nprint("INTEGRIDAD DE DATOS - RESUMEN")\nprint("="*50)\nprint(f"Total de valores NaN en el DataFrame: {nan_totales}\n")\n\nif not nan_top10.empty:\n    print("Top 10 columnas con más valores NaN:")\n    print(nan_top10)\nelse:\n    print("No hay valores NaN en ninguna columna.\n")\n\nprint("-"*50)\nprint(f"Número de filas duplicadas: {duplicados}")\nif duplicados > 0:\n    print("Filas duplicadas:")\n    print(filas_duplicadas)\nprint("-"*50)\n\nif len(fechas_faltantes) > 0:\n    print(f"Fechas faltantes ({len(fechas_faltantes)}):")\n    print(fechas_faltantes)\nelse:\n    print("No hay fechas faltantes.\n")\nprint("="*50)\n'

### Integridad de Datos — Resumen

#### Valores faltantes (NaN)

**Total de valores NaN en el DataFrame:** **4854**

**Top 10 columnas con más valores NaN:**

| Price     | Ticker  | NaN Count |
|-----------|---------|-----------|
| Close     | AENA.MC | 523 |
| Adj Close | AENA.MC | 523 |
| Open      | AENA.MC | 523 |
| Volume    | AENA.MC | 523 |
| Low       | AENA.MC | 523 |
| High      | AENA.MC | 523 |
| Adj Close | ABNB    | 16  |
| Adj Close | AAPL    | 16  |
| Adj Close | COST   | 16  |
| Adj Close | ADBE   | 16  |


#### Filas duplicadas

- **Número de filas duplicadas:** **0**

#### Fechas faltantes (6)

```text
2024-03-29
2024-12-25
2025-01-01
2025-04-18
2025-12-25
2026-01-01

#### Idea rápida

- `AENA.MC` concentra **la mayoría de los NaN** faltan datos históricos.
- No hay filas duplicadas 
- Las fechas faltantes coinciden con **festivos de mercado** (totalmente normal en datos financieros).

Antes de imputar o eliminar NaN por ticker, seguiremos haciendo un estudio básico de los datos ya que puede ser muy precipitado antes de calcular los parámetros históricos

A lo largo del preprocesamiento se construyen los parámetros fundamentales necesarios para modelos financieros (por ejemplo, Black–Scholes o Monte Carlo).

### 1. [Filtrado por fecha de valoración](../src/PreprocesamientoActivos.py#L19)


Dado un conjunto de precios históricos:

$$
\{ S_t^{(i)} \}_{t \in \mathcal{T},\; i=1,\dots,N}
$$

donde:
- $t$ representa la fecha  
- $i$ el activo (ticker)

se fija una **fecha de valoración** $t_0$ y se conservan únicamente los datos tales que:

$$
t \leq t_0
$$

Formalmente:

$$
\mathcal{T}_{\text{filtrado}} = \{ t \in \mathcal{T} \mid t \leq t_0 \}
$$

Esto garantiza que **no se usa información futura** en la estimación de parámetros.





### 2. [Selección de *Adjusted Close* y cálculo de $S_0$](../src/PreprocesamientoActivos.py#L25)

Se selecciona el precio ajustado (*Adjusted Close*), que incorpora dividendos, *splits* y otras acciones corporativas.

Para cada activo $i$, se define el **precio inicial** como:

$$
S_0^{(i)} = S_{t_0}^{(i)}
$$

donde:
- $S_{t_0}^{(i)}$ es el último precio ajustado disponible antes (o en) la fecha de valoración.

Este valor es el **estado inicial del activo** para cualquier modelo estocástico posterior.

### 3. [Cálculo de retornos logarítmicos históricos](../src/PreprocesamientoActivos.py#L37)

Sea una ventana de observación de longitud $n$ (por ejemplo, los últimos 100 precios disponibles).

Para cada activo $i$, los **retornos logarítmicos diarios** [(Fintai, 2026)](https://www.fintai.es/matematicas-financieras/que-son-los-retornos-logaritmicos/) se definen como:

$$
r_t^{(i)} = \ln\left( \frac{S_t^{(i)}}{S_{t-1}^{(i)}} \right)
$$ 

Propiedades clave:
- Son **aditivos en el tiempo**
- Simetría de los retornos

Solo se calculan si el número de observaciones cumple:

$$
n \geq n_{\min}
$$

donde $n_{\min}$ es el número mínimo de precios requeridos (ej. 30).


### 4. [Estimación de volatilidad histórica anualizada](../src/PreprocesamientoActivos.py#L63)

#### 4.1 Volatilidad diaria

La **volatilidad diaria** se estima como la desviación estándar muestral de los retornos:

$$
\sigma_{\text{daily}}^{(i)} =
\sqrt{\frac{1}{T-1} \sum_{t=1}^{T}
\left(r_t^{(i)} - \bar{r}^{(i)}\right)^2}
$$

donde:
- $T$ es el número de retornos observados  
- $\bar{r}^{(i)}$ es el retorno medio


#### 4.2 Volatilidad anualizada

Asumiendo independencia temporal de los retornos:

$$
\sigma_{\text{annual}}^{(i)} = \sigma_{\text{daily}}^{(i)} \sqrt{D}
$$

donde:
- $D = 252$ es el número típico de días de mercado al año

Esta es la volatilidad utilizada en la mayoría de modelos financieros continuos.




### 5. [Asignación de volatilidad *proxy* (activos con pocos datos)](../src/PreprocesamientoActivos.py#L79)

Para activos con menos de $n_{\min}$ observaciones (en nuestro caso 30 pero podemos modificarlo), la volatilidad no se estima directamente.

En su lugar, se asigna una **volatilidad proxy** basada en activos comparables.

Sea $\mathcal{V}$ el conjunto de activos válidos (activos con los que estamos trabajando). La volatilidad proxy puede definirse como:

- **Promedio**:
$$
\sigma_{\text{proxy}} = \frac{1}{|\mathcal{V}|} \sum_{j \in \mathcal{V}} \sigma^{(j)}
$$

- **Mediana**:
$$
\sigma_{\text{proxy}} = \text{median}\left( \{\sigma^{(j)}\}_{j \in \mathcal{V}} \right)
$$

- **Máximo / Mínimo**:
$$
\sigma_{\text{proxy}} = \max_{j \in \mathcal{V}} \sigma^{(j)}
\quad \text{o} \quad
\min_{j \in \mathcal{V}} \sigma^{(j)}
$$

**Importante**: Está aproximación es cuestionable debido a que el conjunto de activos válidos, la estamos tomando de los activos en los que estamos haciendo el estudio. Para futuros pasos, la idea es que dichos activos pertenezcan al mismo sector o segmentación del mercado u otras propiedades, las cuales puedan compartir políticas o estrategias similares en cuanto dividendos




### 6. [Estudio del Dividend Yield $d$](../src/PreprocesamientoActivos.py)

Para cada activo $i$, se obtienen los dividendos pagados durante el último año mediante navegación web:

$$
\{D_k^{(i)}\}_{k=1}^K
$$

El dividend yield anualizado se define como:

$$
d(i) = \frac{\sum_{k=1}^K D_k(i)}{S_0^{(i)}}
$$

donde:

- $S_0^{(i)}$ es el precio inicial del activo (que ya lo conocemos)

De [FullRatio.com](https://fullratio.com/) solo hemos podido extraer los dividendos de estos activos:**AAPL,NVDA,COST,GEHC,META,PEP**

De [estrategiasdeinversion.com](https://www.estrategiasdeinversion.com/cotizaciones/dividendos) hemos podido obtener los dividendos de los activos del IBEX35 (el de todos los activos estudiados o abordados en este proyecto). **IMPORTANTE**: En el csv que explicaremos más adelante contiene los dividendos que ha tenido cada activo a lo largo de 2025, en esta fuente nos dan los dividendos según un periodo. Sin embargo, en todos los activos solo hemos observado un dato `dividendo` para cada activo y hemos supuesto que es anual (simplificación) (aunque ponga trimestral Semianual...)

Nos faltarían los siguientes activos del NASDAQ100: **ADBE,INTC,WBD,TSLA,ABNB,MNST,NFLX**, para hacer el estudio de estos activos hemos buscado a través de [NASDAQ.com](https://www.nasdaq.com/market-activity/stocks/mnst/dividend-history), y nos aparece que el historial de dividendos actualmente no se puede acceder, debido a que la empresa todavía no ha otorgado dividendos, también nos ha aparecido que los últimos dividendos concedidos son anteriores a 2025. Debido a esta información decidimos asignar a todos los activos **0.00** (sin dividendos 2025 confirmados)


Hemos creado un fichero llamado [dividends.csv](../data/dividends.csv), donde se guarda la información extraida de arriba.




In [9]:
from PreprocesamientoActivos import PreprocesamientoActivos

prep = PreprocesamientoActivos(df_hist)
resumen = prep.procesar_pipeline("2026-01-21")
print(resumen)



⚠️ Ticker AENA.MC tiene menos de 30 precios.
{'S0': Ticker
AAPL       247.649994
ABNB       133.589996
ACX.MC      12.890000
ADBE       294.230011
AENA.MC     25.200001
ANA.MC     176.600006
BBVA.MC     20.860001
COST       982.859985
ELE.MC      30.309999
FER.MC      56.680000
GEHC        81.099998
IBE.MC      18.334999
IDR.MC      53.849998
INTC        54.250000
ITX.MC      55.459999
LOG.MC      30.780001
META       612.960022
MNST        81.599998
NFLX        85.360001
NVDA       183.320007
PEP        146.740005
REP.MC      16.139999
SAN.MC      10.324000
SCYR.MC      3.918000
TEF.MC       3.236000
TSLA       431.440002
WBD         28.530001
Name: 2026-01-21 00:00:00, dtype: float64, 'log_returns':                 AAPL      ABNB    ACX.MC      ADBE    ANA.MC   BBVA.MC  \
Date                                                                     
2024-01-02       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-03       NaN       NaN       NaN       NaN       NaN       

In [7]:
# Supongamos que tienes tu df_options y S0_dict listos
from PreprocesamientoOpciones import PreprocesamientoOpciones

df = pd.read_csv("../data/option_chains_all.csv")
proc_opciones = PreprocesamientoOpciones(df, S0)
proc_opciones.convertir_fechas()
proc_opciones.filtrar_vencimiento([21, 52])
proc_opciones.separar_call_put()
proc_opciones.seleccionar_atm()
proc_opciones.calcular_precio_mercado()
proc_opciones.filtrar_liquidez()
proc_opciones.limpiar_nans()
resumen = proc_opciones.resumen()
print(resumen)


{'num_opciones': 6, 'tickers': array(['AAPL', 'NVDA', 'INTC', 'TSLA', 'META', 'NFLX'], dtype=object), 'fechas_expiry': array([Timestamp('2026-08-21 00:00:00'), Timestamp('2027-03-19 00:00:00')],
      dtype=object)}
